<a href="https://colab.research.google.com/github/Xcelrator0/Intership-Tasks/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
if not os.path.isdir("Intership-Tasks"):
    !git clone https://github.com/Xcelrator0/Intership-Tasks.git
%cd Intership-Tasks
%pip install -q pandas numpy scikit-learn

Cloning into 'Intership-Tasks'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 156 (delta 60), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 1.87 MiB | 6.46 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/Intership-Tasks


In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label — never a feature, and neither is trend_pct (same column it's derived from).
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

LABEL_SOURCE_COLS = {"trend_direction", "trend_pct", "is_declining_label"}
ID_COLS = {"content_id", "client_id"}

# Flagged as worth checking yourself before you trust them as features (see note above).
SUSPECT_LEAKAGE_COLS = {
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
}

EXCLUDE = LABEL_SOURCE_COLS | ID_COLS | SUSPECT_LEAKAGE_COLS  # now excluded — see leakage trap below

feature_cols = [c for c in df.columns if c not in EXCLUDE]
numeric_cols = [c for c in feature_cols if df[c].dtype != "object"]
categorical_cols = [c for c in feature_cols if df[c].dtype == "object"]

print("numeric features:", numeric_cols)
print("categorical features:", categorical_cols)
print("\nSuspect leakage columns still included — verify against docs/data-dictionary.md:")
print(SUSPECT_LEAKAGE_COLS & set(feature_cols))

numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
categorical features: ['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

Suspect leakage columns still included — verify against docs/data-dictionary.md:
set()


## 1. Method choice and why
Gradient Boosting is the primary choice because it achieves the highest performance with a Precision@50 of 0.80, significantly outperforming Random Forest (0.68), Logistic Regression (0.64), and single Decision Trees (0.38). Non-linear tree ensembles fit this lane best because content decline relies on complex feature interactions between traffic history, content age, and search positioning that linear models fail to fully capture

## 2. Split design


In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx], df.iloc[test_idx]

overlap = set(train["client_id"]) & set(test["client_id"])
print(f"train rows: {len(train)}, test rows: {len(test)}, client overlap: {len(overlap)} (should be 0)")

train rows: 23837, test rows: 6163, client overlap: 0 (should be 0)


A client-grouped split (GroupShuffleSplit on client_id) ensures zero client overlap between training and test sets. This design prevents data leakage from shared domain authority and client-level baseline habits, forcing the model to demonstrate honest performance on completely unseen clients rather than memorizing client-specific signals.

## 3. Train + compare vs my baseline

In [4]:
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
])

models = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "decision_tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

y_train, y_test = train["is_declining_label"], test["is_declining_label"]
results = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", model)])
    pipe.fit(train[feature_cols], y_train)
    scores = pipe.predict_proba(test[feature_cols])[:, 1]
    results.append({"model": name, "precision_at_50": precision_at_k(y_test, scores, 50)})
    fitted[name] = pipe

# Your Week-4 baseline, for the same test rows, same metric.
try:
    baseline_csv = pd.read_csv("work/outputs/baseline_action_score.csv")
    baseline_test = baseline_csv[baseline_csv["content_id"].isin(test["content_id"])]
    baseline_test = baseline_test.merge(test[["content_id", "is_declining_label"]], on="content_id")
    baseline_p50 = precision_at_k(baseline_test["is_declining_label"], baseline_test["score"].values, 50)
    results.append({"model": "week4_baseline", "precision_at_50": baseline_p50})
except FileNotFoundError:
    print("No baseline CSV found yet — run your w04 notebook first to generate work/outputs/baseline_action_score.csv")

comparison = pd.DataFrame(results).sort_values("precision_at_50", ascending=False)
comparison

No baseline CSV found yet — run your w04 notebook first to generate work/outputs/baseline_action_score.csv


,model,precision_at_50
3,gradient_boosting,0.80
2,random_forest,0.68
0,logistic_regression,0.64
1,decision_tree,0.38


## 4. Errors and interpretation

In [5]:
# Permutation importance for whichever model scored best above.
BEST_MODEL = comparison.iloc[0]["model"]  # auto-picks top row; override manually if you prefer a simpler model
if BEST_MODEL in fitted:
    pi = permutation_importance(fitted[BEST_MODEL], test[feature_cols], y_test, n_repeats=10, random_state=42, scoring="roc_auc")
    importance_df = pd.DataFrame({"feature": feature_cols, "importance": pi.importances_mean}).sort_values("importance", ascending=False)
    print(f"Top features for {BEST_MODEL}:")
    print(importance_df.head(10))
else:
    print(f"{BEST_MODEL} is the baseline, not a fitted model — pick a model row instead, e.g. BEST_MODEL='random_forest'")

Top features for gradient_boosting:
                  feature  importance
18  days_with_impressions    0.100227
20       content_age_days    0.021504
28           avg_position    0.021261
27                    ctr    0.014547
30            scroll_rate    0.007993
11             clicks_90d    0.005628
10        impressions_90d    0.004824
29        engagement_rate    0.003769
6              word_count    0.003121
15   engaged_sessions_90d    0.003084


In [6]:
# False positives / false negatives at the top-50 cut, for your error read.
if BEST_MODEL in fitted:
    scores = fitted[BEST_MODEL].predict_proba(test[feature_cols])[:, 1]
    top50_idx = np.argsort(-scores)[:50]
    top50 = test.iloc[top50_idx].copy()
    top50["predicted_score"] = scores[top50_idx]
    false_positives = top50[top50["is_declining_label"] == 0]
    print(f"{len(false_positives)} of the top 50 picks were NOT actually declining.")
    false_positives[["content_id", "predicted_score", "content_type", "main_intent"]].head(10)

10 of the top 50 picks were NOT actually declining.


The model leans heavily on consistency and search visibility features, driven primarily by days_with_impressions, content_age_days, and avg_position. However, 10 out of the top 50 flagged candidates are false positives (a 20% error rate). The model tends to misclassify stable or niche pages with low historical impression frequency as declining content simply because their activity metrics resemble decaying organic search trends.